# 🩻 Predict Thử Nghiệm Mẫu Thật & So Sánh Trực Quan GT vs Predict (Google Colab)

Notebook này gọi trực tiếp script Python nguồn tại **`tools/predict_real_samples.py`** trong repository GitHub.

### 📋 Quy Trình Thực Hiện:
1. **Kiểm tra phần cứng & Cài đặt môi trường** (PyTorch 2.6.0+cu126, Triton 3.3.1, SimpleITK, `tqdm`, `tabulate`, `matplotlib` + CUDA wheels trên Drive).
2. **Clone / Pull mã nguồn từ GitHub** để đảm bảo luôn dùng phiên bản script mới nhất.
3. **Nạp Mô hình + `slices*.csv` của chính đợt train + `NUM_SAMPLES` mẫu chưa từng train**:
   - Tự động đối chiếu với `slices*.csv` đi kèm run đó để **tách biệt tuyệt đối ca `TRAIN`**.
   - Chỉ copy đúng số lượng mẫu chỉ định (`Val` hoặc `Unseen` hoặc `Custom`) từ Google Drive về SSD `/content` có thanh tiến trình.
4. **Suy luận 3D theo bộ tiêu chí xét ngưỡng lâm sàng**:
   - Ngưỡng độ tin cậy xác suất pixel (`THRESHOLD`) & đỉnh khối (`MIN_COMP_MAX_PROB`).
   - Lọc kích thước nốt theo **đường kính (`MIN_DIAMETER_MM`: dưới 5mm, dưới 10mm...)**, diện tích lát (`MIN_AREA_MM2`) & thể tích (`MIN_VOLUME_MM3`).
   - Thống kê rõ cụm nào **ĐƯỢC GIỮ LẠI** và cụm nào **BỊ LOẠI BỎ** cùng lý do cụ thể.
5. **So sánh từng lát cắt & Vẽ Mask trực quan**:
   - Lấy toàn bộ lát cắt được predict có nốt u hợp với các lát có GT thật để chấm `% chuẩn` (Slice Dice, IoU, Precision, Recall) & các chỉ số 3D (Dice 3D, IoU 3D, HD95 mm, ASSD mm, Volumetric Similarity).
   - Bảng chú giải màu trên ảnh Zoom ROI: 🟩 **Xanh lá = TP (Trúng chuẩn)** | 🟥 **Đỏ = FP (Đoán thừa/nhầm)** | 🟦 **Xanh dương = FN (Bỏ sót GT)**.

---

In [ ]:
# 1. Kiểm tra GPU khả dụng trên Colab (nếu không có GPU sẽ tự chuyển sang fallback CPU)
!nvidia-smi || echo '⚠️ Không có GPU NVIDIA, notebook sẽ chạy ở chế độ fallback CPU.'

In [ ]:
# 2. Kiểm tra & Cài đặt môi trường chạy (PyTorch, Triton, thư viện y tế & biểu đồ)
import shutil, subprocess, sys

def probe(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"🔍 Trạng thái GPU: {'CÓ GPU NVIDIA' if has_gpu else 'KHÔNG CÓ GPU (dùng CPU fallback)'}")

if has_gpu:
    TORCH_PROBE = "import torch;print(torch.__version__, int(torch._C._GLIBCXX_USE_CXX11_ABI))"
    cur = probe(TORCH_PROBE)
    print("Torch hiện tại:", cur)
    if cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"):
        print("⏭️ Đã có torch 2.6.0+cu126 (ABI=True), bỏ qua bước cài đặt torch.")
    else:
        print("🧹 Cài đặt PyTorch 2.6.0+cu126 tương thích với CUDA wheels...")
        !rm -rf /usr/local/lib/python3.*/dist-packages/~* 2>/dev/null || true
        !pip uninstall torch torchvision torchaudio causal-conv1d mamba-ssm -y -q 2>/dev/null || true
        !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126 -q
        cur = probe(TORCH_PROBE)
        print("Torch sau khi cài:", cur)
    assert cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"), \
        f"❌ Torch chưa đúng (cần 2.6.0+cu126 ABI=1, đang là: {cur})"
    !python -m pip install triton==3.3.1 SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib seaborn tqdm tabulate -q
else:
    !python -m pip install SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib seaborn tqdm tabulate -q

print("\n✅ CELL 2 XONG. Chạy tiếp Cell 3 để Mount Google Drive và cài CUDA wheels (nếu dùng GPU).")

In [ ]:
# 3. Mount Google Drive & Cài đặt 2 CUDA Wheels (causal_conv1d + mamba_ssm) nếu chưa có
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

print("🔗 Đang kết nối Google Drive...")
drive.mount('/content/drive')

SOURCES_CANDIDATES = [
    "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/CT208E/SOURCES",
    "/content/drive/MyDrive",
]
DRIVE_SOURCES_DIR = next((d for d in SOURCES_CANDIDATES if os.path.isdir(d)), None)
assert DRIVE_SOURCES_DIR is not None, "❌ Không tìm thấy thư mục dự án trên Google Drive!"
print(f"🎯 Thư mục dự án trên Drive: {DRIVE_SOURCES_DIR}")

RESTART_FLAG = Path("/tmp/.mamba_predict_need_restart")
if RESTART_FLAG.exists():
    RESTART_FLAG.unlink()

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if not has_gpu:
    print("ℹ️ Đang chạy CPU -> bỏ qua cài đặt CUDA wheels (sẽ sử dụng PyTorch scan fallback).")
else:
    probe_mamba = subprocess.run(
        [sys.executable, "-c", "import causal_conv1d_cuda, selective_scan_cuda; print('READY')"],
        capture_output=True, text=True,
    )
    if probe_mamba.returncode == 0 and "READY" in probe_mamba.stdout:
        print("⏭️ Runtime hiện tại đã sẵn sàng causal_conv1d_cuda & selective_scan_cuda! Có thể bỏ qua Cell 4.")
    else:
        WHEEL_CANDIDATES = [f"{DRIVE_SOURCES_DIR}/wheels", "/content/drive/MyDrive/wheels"]
        WHEEL_DIR = next((d for d in WHEEL_CANDIDATES if os.path.isdir(d)), None)
        if WHEEL_DIR:
            import glob
            abi_r = subprocess.run([sys.executable, "-c", "import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')"], capture_output=True, text=True)
            abi = abi_r.stdout.strip()
            cp_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
            CAUSAL_WHL = f"{WHEEL_DIR}/causal_conv1d-1.7.0+cu12torch2.6cxx11abi{abi}-{cp_tag}-{cp_tag}-linux_x86_64.whl"
            MAMBA_WHL  = f"{WHEEL_DIR}/mamba_ssm-2.3.2.post1+cu12torch2.6cxx11abi{abi}-{cp_tag}-{cp_tag}-linux_x86_64.whl"
            if not os.path.exists(CAUSAL_WHL):
                cands = sorted(glob.glob(f"{WHEEL_DIR}/causal_conv1d*cxx11abi{abi}*.whl"))
                CAUSAL_WHL = cands[0] if cands else CAUSAL_WHL
            if not os.path.exists(MAMBA_WHL):
                cands = sorted(glob.glob(f"{WHEEL_DIR}/mamba_ssm*cxx11abi{abi}*.whl"))
                MAMBA_WHL = cands[0] if cands else MAMBA_WHL
            if os.path.exists(CAUSAL_WHL) and os.path.exists(MAMBA_WHL):
                print(f"🚀 Đang cài 2 CUDA wheels từ {WHEEL_DIR}...")
                !python -m pip install "{CAUSAL_WHL}" --force-reinstall --no-deps -q
                !python -m pip install "{MAMBA_WHL}" --force-reinstall --no-deps -q
                RESTART_FLAG.write_text("1")
                print("✅ Đã cài xong 2 CUDA wheels! Hãy chạy Cell 4 để khởi động lại Runtime nạp C++/CUDA.")
            else:
                print(f"⚠️ Không thấy đủ 2 file wheel cho ABI={abi} trong {WHEEL_DIR}. Sẽ dùng PyTorch fallback.")
        else:
            print("⚠️ Không thấy thư mục wheels trên Drive. Sẽ dùng PyTorch fallback an toàn.")

In [ ]:
# 4. Khởi động lại Runtime (CHỈ khởi động lại nếu Cell 3 vừa cài đặt mới CUDA wheels)
import os
from pathlib import Path

RESTART_FLAG = Path("/tmp/.mamba_predict_need_restart")
if RESTART_FLAG.exists():
    RESTART_FLAG.unlink()
    print("🔄 Đang khởi động lại Runtime để nạp kernel C++/CUDA... (Colab báo session crashed là BÌNH THƯỜNG, hãy chạy tiếp Cell 5 ngay sau đó)")
    os.kill(os.getpid(), 9)
else:
    print("✅ Môi trường đã sẵn sàng, KHÔNG cần khởi động lại Runtime. Chạy tiếp Cell 5.")

In [ ]:
# 5. Kiểm tra xác minh môi trường PyTorch & Mamba-SSM (CUDA / Fallback)
import sys, torch

print("python :", f"cp{sys.version_info.major}{sys.version_info.minor}")
print("torch  :", torch.__version__, "| ABI =", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", False))
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ (CPU Fallback)")

if torch.cuda.is_available():
    try:
        import causal_conv1d_cuda, selective_scan_cuda
        print("mamba  : ✅ CUDA native selective_scan khả dụng (suy luận siêu nhanh).")
    except Exception as e:
        print(f"mamba  : ⚠️ Không nạp được CUDA kernel ({e}) -> Tự động dùng PyTorch fallback.")
else:
    print("mamba  : ℹ️ Chạy PyTorch scan fallback trên CPU.")

In [ ]:
# 6. Clone & Đồng bộ mã nguồn từ GitHub về /content
import os, sys
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('tools/predict_real_samples.py') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Repository đã có sẵn, đang cập nhật nhánh {BRANCH}...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 6b. Cell PULL nhanh mã nguồn từ GitHub khi bạn vừa có chỉnh sửa nhỏ
BRANCH = 'main'
%cd /content/CT208E_LIDC_DuneAI_MambaVision
!git fetch origin
!git pull origin {BRANCH}
!git log -1 --oneline

In [ ]:
# 7. Nạp Mô hình + slices*.csv của chính đợt train + Đúng N mẫu chưa từng train từ Drive lên /content
import os

# Dò lại DRIVE_SOURCES_DIR nếu runtime vừa được restart ở Cell 4
SOURCES_CANDIDATES = [
    "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/CT208E/SOURCES",
    "/content/drive/MyDrive",
]
DRIVE_SOURCES_DIR = next((d for d in SOURCES_CANDIDATES if os.path.isdir(d)), None)
assert DRIVE_SOURCES_DIR is not None, "❌ Chưa mount Google Drive hoặc không thấy thư mục dự án!"

# =============================================================================
# CẤU HÌNH NẠP MÔ HÌNH & CHỌN MẪU CHƯA TRAIN
# =============================================================================
ARCH = "mamba_unet"                  # Kiến trúc mô hình Mamba-UNet
TARGET_RUN = "r2bc"                  # Preset: 'r2bc', 'r2a', 'r2b', 'r2c', 'r1c', 'r1b', 'r1a', 'r0b', hoặc 'custom'
CUSTOM_RUN_OR_CKPT = ""              # Nếu TARGET_RUN='custom', điền đường dẫn thư mục run hoặc file .pth trên Drive
CKPT_NAME = "best_model.pth"         # Hoặc 'best_weights_fp16.pth' / 'last_model.pth'

# Nguồn bệnh nhân đảm bảo KHÔNG nằm trong tập Train của đợt chạy trên:
#   - 'val'      : Các ca nằm trong split=='val' của file slices*.csv đợt train (vd 75 ca Val)
#   - 'unseen'   : Các ca hoàn toàn mới trên Drive KHÔNG có mặt trong slices*.csv đợt train (vd ca 0501..1012)
#   - 'non_train': Gộp cả 'val' + 'unseen' (loại sạch toàn bộ ca 'train')
#   - 'custom'   : Tự chỉ định danh sách ở CUSTOM_PATIENT_IDS bên dưới (vẫn chặn nếu dính ca train!)
SAMPLE_SOURCE = "val"
SELECTION_MODE = "mixed"             # 'mixed' (gồm cả ca có u + ca sạch), 'tumor_first', 'first', 'random'
NUM_SAMPLES = 5                      # CHỈ tải đúng số bệnh nhân chỉ định này lên SSD /content (0 = toàn bộ tập nguồn)
CUSTOM_PATIENT_IDS = []              # vd: ["LIDC-IDRI-0051", "LIDC-IDRI-0089"] khi SAMPLE_SOURCE='custom'
ALLOW_TRAIN_LEAK = False             # True chỉ khi muốn cố tình test lại trên ca TRAIN

DATA_SOURCE = "auto"                 # 'auto' (tự tìm thư mục cache_1mm hoặc cache_1mm.zip trên Drive)
DEST_PREDICT_CACHE = "/content/data/predict_cache_1mm"
LOCAL_FAST_CACHE = "/content/data/cache_1mm"  # Nếu phiên Colab đã copy sẵn cache ở đây thì lấy tức thì!
COPY_WORKERS = 6
SEED = 42

# =============================================================================
# GỌI SCRIPT PYTHON CHUẨN BỊ DỮ LIỆU
# =============================================================================
prep_parts = [
    "python -u tools/predict_real_samples.py prepare",
    f"--drive-sources-dir '{DRIVE_SOURCES_DIR}'",
    f"--arch {ARCH}",
    f"--ckpt-name {CKPT_NAME}",
    f"--data-source '{DATA_SOURCE}'",
    f"--local-fast-cache '{LOCAL_FAST_CACHE}'",
    f"--dest-cache '{DEST_PREDICT_CACHE}'",
    f"--source {SAMPLE_SOURCE}",
    f"--select {SELECTION_MODE}",
    f"--num-samples {NUM_SAMPLES}",
    f"--workers {COPY_WORKERS}",
    f"--seed {SEED}",
]
if TARGET_RUN == "custom" or CUSTOM_RUN_OR_CKPT.strip():
    assert CUSTOM_RUN_OR_CKPT.strip(), "❌ Khi TARGET_RUN='custom' phải điền CUSTOM_RUN_OR_CKPT!"
    prep_parts.append(f"--run-dir '{CUSTOM_RUN_OR_CKPT.strip()}'")
else:
    prep_parts.append(f"--run-preset {TARGET_RUN}")

if SAMPLE_SOURCE == "custom" and CUSTOM_PATIENT_IDS:
    pids_str = " ".join(CUSTOM_PATIENT_IDS)
    prep_parts.append(f"--patient-ids {pids_str}")
if ALLOW_TRAIN_LEAK:
    prep_parts.append("--allow-train-leak")

prep_cmd = " \\\n    ".join(prep_parts)
print(f"🚀 Lệnh thực thi:\n{prep_cmd}\n" + "=" * 80)
!{prep_cmd}

In [ ]:
# 8. Cấu hình Tiêu chí Ngưỡng & Chạy Predict Mẫu Thật + So Sánh GT
import json, os
from pathlib import Path

MANIFEST_PATH = Path("/content/data/predict_cache_1mm/predict_manifest.json")
assert MANIFEST_PATH.exists(), "❌ Chưa thấy predict_manifest.json! Hãy chạy Cell 7 trước."
manifest_data = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
run_name = Path(manifest_data["model_checkpoint"]).parent.name
print("📋 Các bệnh nhân đã nạp sẵn từ Cell 7:", manifest_data.get("selected_ids", []))

# =============================================================================
# CHỌN BỆNH NHÂN ĐỂ PREDICT & CẤU HÌNH TIÊU CHÍ XÉT NGƯỠNG
# =============================================================================
# Để [] = chạy toàn bộ các ca đã chọn ở Cell 7; hoặc điền đích danh, vd: ["LIDC-IDRI-0051"]
PATIENT_IDS = []
MAX_PATIENTS = 0            # 0 = không giới hạn; > 0 = chỉ lấy N ca đầu tiên

THRESHOLD = 0.5             # Ngưỡng độ tin cậy xác suất mỗi pixel [0.0 .. 1.0] (vd: 0.5, 0.6, 0.7)
MIN_COMP_MAX_PROB = 0.0     # Ngưỡng xác suất đỉnh tối thiểu của cả khối u (vd: 0.6 để loại đốm mờ)
MIN_DIAMETER_MM = 5.0       # Ngưỡng ĐƯỜNG KÍNH (mm): 0.0=lấy mọi kích thước | 5.0=bỏ dưới 5mm | 10.0=bỏ dưới 10mm
MIN_AREA_MM2 = 5.0          # Ngưỡng DIỆN TÍCH lát cắt tối thiểu (mm²) (vd: 5.0, 10.0, 20.0)
MIN_VOLUME_MM3 = 10.0       # Ngưỡng THỂ TÍCH 3D tối thiểu (mm³) (vd: 0.0, 10.0, 20.0)
MIN_SLICES_SPAN = 1         # Số lát cắt liên tiếp tối thiểu dọc trục Z (1 hoặc 2)
FILTER_SCOPE = "3d_component" # '3d_component' (lọc theo khối u 3D) | 'per_slice' (lọc từng lát 2D) | 'both'

TTA = False                 # True: Bật Test-Time Augmentation (lật ngang/dọc khi predict)
BATCH_SIZE = 16             # Tăng lên 16 hoặc 32 khi inference trên L4/A100 để chạy nhanh hơn
MAX_VIS_SLICES = 0          # 0 = Vẽ TẤT CẢ các lát cắt được predict có u hoặc có GT thật
SLICES_PER_PAGE = 6         # Số lát cắt trên 1 trang ảnh PNG để dễ nhìn
SYNC_TO_DRIVE = True        # True: Tự động đồng bộ ảnh & báo cáo sang Google Drive

SOURCES_CANDIDATES = [
    "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES",
    "/content/drive/MyDrive/CT208E/SOURCES",
    "/content/drive/MyDrive",
]
drive_dir = globals().get("DRIVE_SOURCES_DIR") or next((d for d in SOURCES_CANDIDATES if os.path.isdir(d)), "")
OUT_BASE_DIR = f"eval_results/predict_samples/{run_name}"
DRIVE_OUT_BASE_DIR = f"{drive_dir}/eval_results/predict_samples/{run_name}" if (SYNC_TO_DRIVE and drive_dir) else ""

pred_parts = [
    "python -u tools/predict_real_samples.py predict",
    f"--manifest '{MANIFEST_PATH}'",
    f"--batch-size {BATCH_SIZE}",
    f"--threshold {THRESHOLD}",
    f"--min-comp-max-prob {MIN_COMP_MAX_PROB}",
    f"--min-diameter-mm {MIN_DIAMETER_MM}",
    f"--min-area-mm2 {MIN_AREA_MM2}",
    f"--min-vol-mm3 {MIN_VOLUME_MM3}",
    f"--min-slices-span {MIN_SLICES_SPAN}",
    f"--filter-scope {FILTER_SCOPE}",
    f"--slices-per-page {SLICES_PER_PAGE}",
    f"--max-vis-slices {MAX_VIS_SLICES}",
    f"--out-dir '{OUT_BASE_DIR}'",
    "--auto-subdir",
]
if PATIENT_IDS:
    pred_parts.append(f"--patient-ids {' '.join(PATIENT_IDS)}")
if MAX_PATIENTS > 0:
    pred_parts.append(f"--max-patients {MAX_PATIENTS}")
if TTA:
    pred_parts.append("--tta")
if DRIVE_OUT_BASE_DIR:
    pred_parts.append(f"--drive-out-dir '{DRIVE_OUT_BASE_DIR}'")
if globals().get("ALLOW_TRAIN_LEAK", False):
    pred_parts.append("--allow-train-leak")

pred_cmd = " \\\n    ".join(pred_parts)
print(f"🚀 Khởi chạy Predict mẫu thật:\n{pred_cmd}\n" + "=" * 80)
!{pred_cmd}

latest_ptr = Path(OUT_BASE_DIR) / "latest_predict_dir.txt"
OUT_DIR = latest_ptr.read_text(encoding="utf-8").strip() if latest_ptr.exists() else OUT_BASE_DIR
print(f"\n📂 Thư mục kết quả của lượt chạy này: {OUT_DIR}")

In [ ]:
# 9. Hiển thị Trực Quan Bảng Kết Quả & Ảnh So Sánh Mask Predict vs GT Ngay Trong Colab
import os
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown

out_str = globals().get("OUT_DIR")
if out_str:
    out_path = Path(out_str)
else:
    ptrs = sorted(Path("eval_results/predict_samples").glob("*/latest_predict_dir.txt"), key=os.path.getmtime)
    if ptrs:
        out_path = Path(ptrs[-1].read_text(encoding="utf-8").strip())
    else:
        cands = sorted(Path("eval_results/predict_samples").glob("*/*"), key=os.path.getmtime)
        assert cands, "❌ Chưa có thư mục kết quả predict nào! Hãy chạy Cell 8 trước."
        out_path = cands[-1]
assert out_path.exists(), f"❌ Không thấy thư mục kết quả {out_path}. Hãy chạy Cell 8 trước."
print(f"📁 Đang đọc kết quả từ: {out_path}")

df_pat = pd.read_csv(out_path / "predict_patients_summary.csv")
df_nod = pd.read_csv(out_path / "predict_nodules_detail.csv")
df_slc = pd.read_csv(out_path / "predict_slices_detail.csv")

display(Markdown("### 📊 1. Bảng Tổng Hợp Mức Bệnh Nhân (Độ chuẩn 3D & Lát cắt)"))
display(df_pat)

if len(df_nod):
    display(Markdown("### 🎯 2. Chi Tiết Từng Khối U 3D (GT Thật & Cụm Predict Giữ lại / Đã lọc bỏ)"))
    display(df_nod)

if len(df_slc):
    display(Markdown("### 🩻 3. Bảng Đối Chiếu Từng Lát Cắt (Lát Predict vs GT)"))
    display(df_slc)

fig_dir = out_path / "figures"
if fig_dir.exists():
    display(Markdown("### 🖼️ 4. Hình Ảnh So Sánh Mask (🟩 TP = Khớp | 🟥 FP = Đoán thừa | 🟦 FN = Bỏ sót)"))
    for pid in df_pat["patient_id"]:
        display(Markdown(f"---\n#### Bệnh nhân `{pid}`"))
        ov_img = fig_dir / f"{pid}_3d_overview.png"
        if ov_img.exists():
            display(Image(filename=str(ov_img)))
        for p_img in sorted(fig_dir.glob(f"{pid}_slices_p*.png")):
            display(Image(filename=str(p_img)))